In [14]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertForNextSentencePrediction, BertTokenizer

In [45]:
from skt.vault_utils import get_secrets

proxies = get_secrets("proxies")
import os

import numpy as np

os.environ["http_proxy"] = proxies["http"]
os.environ["https_proxy"] = proxies["https"]

In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [5]:
model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [6]:
def retrieve_next_sentence(current_sentence):
    # Tokenize input
    tokenized_input = tokenizer.encode_plus(
        current_sentence, text_pair=None, add_special_tokens=True, return_tensors="pt"
    )

    # Obtain predictions
    outputs = model(**tokenized_input)
    probs = torch.softmax(outputs.logits, dim=-1)
    next_sentence_prob = probs[:, 0].item()

    # Select next sentence
    if next_sentence_prob > 0.5:
        next_sentence = "The next sentence."
    else:
        next_sentence = "Not the next sentence."

    return next_sentence


# Example usage
current_sentence = "This is an example sentence."
next_sentence = retrieve_next_sentence(current_sentence)
print("Next Sentence:", next_sentence)

Next Sentence: The next sentence.


In [8]:
# Input sentences
sentence_A = "The quick brown fox jumps over the lazy dog."
sentence_B = "He ran as fast as he could to catch up with it."

# Tokenize input sentences
inputs = tokenizer(sentence_A, sentence_B, return_tensors="pt")

# Predict next sentence probability
outputs = model(**inputs)
probs = outputs.logits

# Get probability for next sentence being the continuation
next_sentence_probability = probs[0][0].item()

print(
    "Confidence score of the second sentence being the next sentence:",
    next_sentence_probability,
)

Confidence score of the second sentence being the next sentence: -2.4058055877685547


In [11]:
import torch.nn.functional as F

sentence_A = "The quick brown fox jumps over the lazy dog."
sentence_B = "He ran as fast as he could to catch up with it."

# Tokenize input sentences
inputs = tokenizer(sentence_A, sentence_B, return_tensors="pt")

# Predict next sentence probability
outputs = model(**inputs)
logits = outputs.logits

# Apply softmax to obtain probabilities
probs = F.softmax(logits, dim=1)

# Get probability for next sentence being the continuation
next_sentence_probability = probs[0][
    1
].item()  # Probability of "sentence_B" being next sentence

print(
    "Confidence score of the second sentence being the next sentence:",
    next_sentence_probability,
)

Confidence score of the second sentence being the next sentence: 0.9995666146278381


In [25]:
import torch.nn.functional as F

sentence_A = "i went to beach"
sentence_B = "when?"

# Tokenize input sentences
inputs = tokenizer(sentence_A, sentence_B, return_tensors="pt")

# Predict next sentence probability
outputs = model(**inputs)
logits = outputs.logits

# Apply softmax to obtain probabilities
probs = F.softmax(logits, dim=1)

# Get probability for next sentence being the continuation
next_sentence_probability = probs[0][
    1
].item()  # Probability of "sentence_B" being next sentence

print(
    "Confidence score of the second sentence being the next sentence:",
    next_sentence_probability,
)

Confidence score of the second sentence being the next sentence: 0.0010549830039963126


In [28]:
# korean
from transformers import AutoTokenizer, BertForNextSentencePrediction

model = BertForNextSentencePrediction.from_pretrained("klue/bert-base")
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [32]:
import torch.nn.functional as F

sentence_A = (
    "2002년 월드컵 축구대회는 일본과 공동으로 개최되었던 세계적인 큰 잔치입니다."
)
sentence_B = "극장에 가자"

# Tokenize input sentences
inputs = tokenizer(sentence_A, sentence_B, return_tensors="pt")

# Predict next sentence probability
outputs = model(**inputs)
logits = outputs.logits

# Apply softmax to obtain probabilities
probs = F.softmax(logits, dim=1)

# Get probability for next sentence being the continuation
next_sentence_probability = probs[0][
    1
].item()  # Probability of "sentence_B" being next sentence
print(probs)
print(
    "Confidence score of the second sentence being the next sentence:",
    next_sentence_probability,
)

tensor([[0.0078, 0.9922]], grad_fn=<SoftmaxBackward0>)
Confidence score of the second sentence being the next sentence: 0.9922497272491455


In [48]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens"
)

In [24]:
sentences = ["This is an example sentence", "Each sentence is converted"]
sentences = ["When you born?", "what time is it?"]

In [19]:
embeddings = model.encode(sentences)

In [20]:
embeddings

array([[ 0.44923988, -0.06809565,  0.52514917, ..., -0.69551754,
         0.12216152, -0.14793341],
       [-0.41660613, -0.710462  ,  1.6371535 , ..., -0.59315073,
         0.62219995, -0.05318163]], dtype=float32)

In [21]:
import numpy as np
from numpy import dot
from numpy.linalg import norm


def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

In [32]:
sentences = ["너야", "나야"]

In [34]:
def cal_two_sent(sentences):
    embeddings = model.encode(sentences)
    return cos_sim(embeddings[0], embeddings[1])

In [35]:
cal_two_sent(sentences)

0.8541133

In [33]:
cos_sim(embeddings[0], embeddings[1])

0.53054667

In [ ]:
sentences = [""]

In [42]:
import requests

In [46]:
path = "http://10.40.92.219:8888/v2/shaker/static/flo_v3/track:464145445"

In [50]:
import requests

url = path
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    # print(data['data'])
    # print(data['data']['track_title'])
    # print(data['data']['value']['lyric_list'][0]['lyric_desc'])

    print(data)
else:
    print("Error:", response.status_code)

Error: 403


In [ ]:
fdfgjhjhj


fgf